# cuVS Python API

This notebook shows some of the capabilities of the cuVS Python API, including:

* Building and searching various ANN indices
* CAGRA/HNSW interop: Building on GPU and Searching on the CPU
* Utilizing preprocessing techniques like scalar quantization to reduce the dataset size

## Load the dataset

We first need to load up the embeddings that were previously created:

In [1]:
%%time
import cupy as cp
import numpy as np

dataset = cp.load("./data/embeddings.npy")
queries = dataset[:1024]
dataset.shape

# cuVS supports multiple different distance metric, but for this notebook we are going to use Euclidean (L2) distance
metric = "sqeuclidean"
k = 10

/opt/conda/envs/cuvs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU times: user 11 s, sys: 6.1 s, total: 17.1 s
Wall time: 10.7 s


## Brute Force KNN

One strategy for doing vector search is to exhaustively calculate the distance from the queries to every single vector in the dataset, and then take the top K closest vectors.  The advantage to brute force knn is that it doesn't require any expensive up front index building, and it returns every possible result - but it has the big downside in that it is too slow for latency critical applications unless the dataset is small.  We are mainly showing this functionality here so that we can generate the ground truth neighbors, so that we can calculate recall for our Approximate Nearest Neighbors (ANN) algorithms that we show next.

cuVS provides Brute Force KNN functionality through the `brute_force` module:


In [2]:
%%time
from cuvs.neighbors import brute_force

# build a brute force index from the dataset
brute_force_index = brute_force.build(dataset, metric=metric)

CPU times: user 86.9 ms, sys: 132 ms, total: 219 ms
Wall time: 3.89 s


In [3]:
%%time
# calculate the ground truth neighbors
_, ground_truth_neighbors = brute_force.search(brute_force_index, queries, k=k)
ground_truth_neighbors = ground_truth_neighbors.copy_to_host()

CPU times: user 672 ms, sys: 8.13 ms, total: 680 ms
Wall time: 717 ms


## CAGRA

We can achieve massive speedups over brute force search by using an Approximate Nearest Neighbors (ANN) strategy.

CAGRA is a graph-based index that is based loosely on the popular navigable small-world graph (NSG) algorithm, but which has been built from the ground-up specifically for the GPU. CAGRA constructs a flat graph representation by first building a kNN graph of the training points and then removing redundant paths between neighbors.

There are various parameters that affect the recall and latency of the ANN search with CAGRA - the [CAGRA documentation](https://docs.rapids.ai/api/cuvs/nightly/indexes/cagra/#build-parameters) has a full list of these parameters - but for this example we are just setting the `graph_degree` parameter when building to control the size of the KNN graph that we are building, the `intermediate_graph_degree` that controls the degree of the graph before optimization, and using NN-Descent for the `build_algo`:

In [4]:
%%time
from cuvs.neighbors import cagra

# build a CAGRA index on the dataset
cagra_index_params = cagra.IndexParams(metric=metric, graph_degree=32, intermediate_graph_degree=96, build_algo="nn_descent")
cagra_index = cagra.build(cagra_index_params, dataset)

CPU times: user 1min 19s, sys: 8.88 s, total: 1min 28s
Wall time: 15.7 s


[   360][19:41:16:144534][info  ] optimizing graph
[   360][19:41:16:956968][info  ] Graph optimized, creating index


Searching with the cuVS python API also takes a [multiple different parameters](https://docs.rapids.ai/api/cuvs/nightly/indexes/cagra/#search-parameters), and we are setting the `itopk_size` parameter here for setting the number of intermediate results to retain during search:

In [5]:
%%timeit
# search the cagra index, getting a matrix of distances and neighbors for each query
cagra_search_params = cagra.SearchParams(itopk_size=128)
cagra_distances, cagra_neighbors = cagra.search(cagra_search_params, cagra_index, queries, k=k)

21.7 ms ± 563 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


For comparison of times on the CPU - please see the supplemental notebook where we compare CAGRA to CPU versions of HNSW.

We can calculate the recall of our CAGRA search by comparing to the exact results we calculated with brute force knn - and can see here that we have retrieved over 99% of the relevant results in a fraction of the amount of time that brute force knn has taken.

In [6]:
from cuvs.tests.ann_utils import calc_recall

_, cagra_neighbors = cagra.search(cagra.SearchParams(itopk_size=64), cagra_index, queries, k=k)
recall = calc_recall(cagra_neighbors.copy_to_host(), ground_truth_neighbors)
recall

0.99619140625

# Exercise: Investigate impact of different CAGRA parameters

As an exercise, try playing with some of `cagra.IndexParams` and `cagra.SearchParams` , with the goal of figuring out how setting these parameters impacts both:

* Recall (from the `calc_recall` function comparing to brute force knn)
* Search Latency (as measured by `%%timeit` annotations)
* Build times

Some parameters to try setting are the `SearchParams.itopk_size` - which increasing the value leads to higher recall and worse search latency, or the `IndexParams.graph_degree` which controls the size of the graph being created.

In [7]:
cagra_search_params = cagra.SearchParams(itopk_size=128)
cagra_index_params = cagra.IndexParams(metric=metric, graph_degree=32, intermediate_graph_degree=96, build_algo="nn_descent")

In [8]:
%%time
cagra_index = cagra.build(cagra_index_params, dataset)

CPU times: user 1min 13s, sys: 9.18 s, total: 1min 23s
Wall time: 15.4 s


[   360][19:41:35:191082][info  ] optimizing graph
[   360][19:41:36:001449][info  ] Graph optimized, creating index


In [9]:
%%timeit
cagra.search(cagra_search_params, cagra_index, queries, k=k) 

21.1 ms ± 116 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [10]:
_, cagra_neighbors = cagra.search(cagra_search_params, cagra_index, queries, k=k) 
recall = calc_recall(cagra_neighbors.copy_to_host(), ground_truth_neighbors)
recall

0.99873046875

# CAGRA/HNSW interop: Building on GPU and Searching on the CPU

One of the capabilities of cuVS is that we can build a CAGRA index on the GPU, but then search this index on the CPU using HNSW.

In [11]:
%%time
from cuvs.neighbors import hnsw

# create a CPU hnsw index from the already built CAGRA index
hnsw_params = hnsw.IndexParams()
hnsw_index = hnsw.from_cagra(hnsw_params, cagra_index)

CPU times: user 1.36 s, sys: 11.1 s, total: 12.4 s
Wall time: 14.4 s


In [12]:
%%time
# Search the HNSW index on the CPU
hnsw_search_params = hnsw.SearchParams()
distances, hnsw_neighbors = hnsw.search(hnsw_search_params, hnsw_index, cp.asnumpy(queries), k)

CPU times: user 21.1 s, sys: 134 ms, total: 21.3 s
Wall time: 749 ms


In [13]:
calc_recall(hnsw_neighbors, ground_truth_neighbors)

0.99892578125

# IVF-Flat

IVF-Flat is an inverted file index (IVF) algorithm, where the dataset is clustered into centroids using k-means - and at search time we do an exhaustive brute force search of a subset of the clusters that are closest to the query. This leads to much faster search performance than brute force knn.

The API for constructing an ivf_flat is nearly identical to CAGRA. Here we are setting the `n_list` parameter to control the number of centroids, but there are more build parameters in the [IVF-Flat Documentation](https://docs.rapids.ai/api/cuvs/nightly/indexes/ivfflat/) ) :

In [14]:
%%time
from cuvs.neighbors import ivf_flat

ivf_flat_index_params = ivf_flat.IndexParams(metric=metric, n_lists=1000)

# build a IVF-Flat index on the dataset
ivf_flat_index = ivf_flat.build(ivf_flat_index_params, dataset)

CPU times: user 1.4 s, sys: 191 ms, total: 1.59 s
Wall time: 1.68 s


Searching the ivf_flat index also takes a parameters structure, and here we are setting the `n_probes` controls how many clusters we  will search:

In [15]:
ivf_flat_search_params = ivf_flat.SearchParams(n_probes=40)

In [16]:
%%timeit

# search the IVF-Flat index, getting a matrix of distances and neighbors for each query
distances, ivf_flat_neighbors = ivf_flat.search(ivf_flat_search_params, ivf_flat_index, queries, k=k) 

223 ms ± 1.72 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
# Calculate the recall for ivf-flat, comparing to the bfknn search
distances, ivf_flat_neighbors = ivf_flat.search(ivf_flat_search_params, ivf_flat_index, queries, k=k) 
calc_recall(ivf_flat_neighbors.copy_to_host(), ground_truth_neighbors)

0.94326171875

# Preprocessing - Scalar Quantization

We can reduce the size of the dataset, by quantizing the data down to a 8-bit integer - and then running our ANN indices on the quantized data. By going from a 32-bit float to a 8-bit integer for storing the embeddings, the dataset memory size is reduced by 4x with only a minimal loss of recall:

In [18]:
cagra_index = hnsw_index = ivf_flat_index = None

In [19]:
%%time
from cuvs.preprocessing.quantize import scalar

cagra_index = hnsw_index = ivf_flat_index = None

# train the scalar quantizer on a subset of the dataset
quantizer = scalar.train(scalar.QuantizerParams(quantile=0.99), dataset[:100_000])

CPU times: user 4.03 ms, sys: 11 ms, total: 15.1 ms
Wall time: 132 ms


In [20]:
%%time
# use the quantizer to transform the input dataset and queries from a float32 representation
# down to an 8 bit integer:
sq_dataset = scalar.transform(quantizer, dataset)
sq_queries = scalar.transform(quantizer, queries)

CPU times: user 3.9 ms, sys: 11 ms, total: 14.9 ms
Wall time: 71.1 ms


In [21]:
# train an ivf-flat index on the quantized data
sq_index = ivf_flat.build(ivf_flat_index_params, sq_dataset)

In [22]:
%%time
# search the quantized index and calculate recall
_, sq_neighbors = ivf_flat.search(ivf_flat_search_params, sq_index, sq_queries, k=k)

CPU times: user 80.4 ms, sys: 10.7 ms, total: 91 ms
Wall time: 261 ms


In [23]:
_, sq_neighbors = ivf_flat.search(ivf_flat_search_params, sq_index, sq_queries, k=k)
recall = calc_recall(sq_neighbors.copy_to_host(), ground_truth_neighbors)
recall

0.9306640625

In [24]:
# Clean up GPU memory
import IPython 
app = IPython.Application.instance()
app.kernel.do_shutdown(restart=False)

{'status': 'ok', 'restart': False}